This code allows to extract water and land related polygons from S-57 file

In [ ]:
import sqlite3
import gzip
import math
import mapbox_vector_tile
import geopandas as gpd
from shapely.geometry import shape


MBTILES_PATH = "../data/routing_hg_20251211.mbtiles"
OUTPUT_PATH = "navigable_water.gpkg"

WATER_LAYERS = [
    "HG_DEPARE", "DEPARE", "RESARE", "DRGARE", "UNSARE",
    "LAKARE", "RIVERS", "CANALS", "HG_LAKARE", 
    "BRIDGE", "SLCONS", "DOCARE"              
]

ZOOM = 12
# Stockholm Bounding Box (min_lon, min_lat, max_lon, max_lat)
BBOX = (17.5, 59.1, 18.8, 59.5)

# Earth's circumference in meters for EPSG:3857
TILE_EXTENT = 4096
WORLD_SIZE = 2 * math.pi * 6378137
ORIGIN_SHIFT = WORLD_SIZE / 2

def lonlat_to_tile(lon, lat, z):
    lat_rad = math.radians(lat)
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1/math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

def get_tile_origin_3857(x, y, z):
    res = WORLD_SIZE / (2**z)
    mx = x * res - ORIGIN_SHIFT
    my = ORIGIN_SHIFT - y * res
    return mx, my, res

def project_mvt_geom(geom, tile_x, tile_y, zoom):
    mx_origin, my_origin, tile_res = get_tile_origin_3857(tile_x, tile_y, zoom)
    unit_res = tile_res / TILE_EXTENT

    def transform_coords(coords):
        return [(mx_origin + (px * unit_res), my_origin - (py * unit_res)) for px, py in coords]

    if geom["type"] == "Polygon":
        return {"type": "Polygon", "coordinates": [transform_coords(r) for r in geom["coordinates"]]}
    elif geom["type"] == "MultiPolygon":
        return {"type": "MultiPolygon", "coordinates": [[transform_coords(r) for r in p] for p in geom["coordinates"]]}
    elif geom["type"] == "LineString":
        return {"type": "LineString", "coordinates": transform_coords(geom["coordinates"])}
    return geom


def extract_vector_data(mbtiles_path, bbox, zoom):
    min_lon, min_lat, max_lon, max_lat = bbox
    x_min, y_max_xyz = lonlat_to_tile(min_lon, min_lat, zoom)
    x_max, y_min_xyz = lonlat_to_tile(max_lon, max_lat, zoom)

    x_min, x_max = sorted([x_min, x_max])
    y_min_xyz, y_max_xyz = sorted([y_min_xyz, y_max_xyz])

    tms_y_min = (1 << zoom) - 1 - y_max_xyz
    tms_y_max = (1 << zoom) - 1 - y_min_xyz

    conn = sqlite3.connect(mbtiles_path)
    cur = conn.cursor()
    cur.execute(f"""
        SELECT tile_column, tile_row, tile_data FROM tiles
        WHERE zoom_level = {zoom}
          AND tile_column BETWEEN {x_min} AND {x_max}
          AND tile_row BETWEEN {tms_y_min} AND {tms_y_max}
    """)

    features = []
    rows = cur.fetchall()
    print(f"Processing {len(rows)} tiles...")

    for x_tile, y_tms, data in rows:
        y_xyz = (1 << zoom) - 1 - y_tms
        tile_pbf = gzip.decompress(data)
        # y_coord_down=True is essential for MVT standard alignment
        decoded = mapbox_vector_tile.decode(tile_pbf, y_coord_down=True)

        for layer_name in WATER_LAYERS:
            if layer_name in decoded:
                for feat in decoded[layer_name]["features"]:
                    merc_geom = project_mvt_geom(feat["geometry"], x_tile, y_xyz, zoom)
                    poly_shape = shape(merc_geom)

                    props = feat["properties"].copy()
                    props["original_layer"] = layer_name

                    features.append({
                        "geometry": poly_shape,
                        **props
                    })

    conn.close()

    if not features:
        print("no features found")
        return None

    print(f"Found {len(features)} total features. Converting to GeoPackage...")
    gdf_3857 = gpd.GeoDataFrame(features, crs="EPSG:3857")
    return gdf_3857.to_crs("EPSG:4326")


water_gdf = extract_vector_data(MBTILES_PATH, BBOX, ZOOM)

if water_gdf is not None:
    water_gdf.to_file(OUTPUT_PATH, driver="GPKG", layer="navigable_water")

In [ ]:

MBTILES_PATH = "../data/routing_hg_20251211.mbtiles"
OUTPUT_PATH = "land_stockholm_final.gpkg"
LAYER_NAME = "LNDARE_POLYGON"


def extract_land_data(mbtiles_path, bbox, zoom):
    min_lon, min_lat, max_lon, max_lat = bbox

    # 1. Get XYZ Tile Range
    x_min, y_max_xyz = lonlat_to_tile(min_lon, min_lat, zoom)
    x_max, y_min_xyz = lonlat_to_tile(max_lon, max_lat, zoom)

    # Sort to ensure range is correct
    x_min, x_max = sorted([x_min, x_max])
    y_min_xyz, y_max_xyz = sorted([y_min_xyz, y_max_xyz])

    # 2. Convert XYZ to TMS for the SQLite Query
    # MBTiles stores Y from the Bottom-Up
    tms_y_min = (1 << zoom) - 1 - y_max_xyz
    tms_y_max = (1 << zoom) - 1 - y_min_xyz

    print(f"Querying Tiles: X[{x_min}-{x_max}] | TMS_Y[{tms_y_min}-{tms_y_max}]")

    conn = sqlite3.connect(mbtiles_path)
    cur = conn.cursor()
    cur.execute(f"""
        SELECT tile_column, tile_row, tile_data
        FROM tiles
        WHERE zoom_level = {zoom}
          AND tile_column BETWEEN {x_min} AND {x_max}
          AND tile_row BETWEEN {tms_y_min} AND {tms_y_max}
    """)

    features = []
    rows = cur.fetchall()

    for x_tile, y_tms, data in rows:
        y_xyz = (1 << zoom) - 1 - y_tms

        tile_pbf = gzip.decompress(data)

        decoded = mapbox_vector_tile.decode(tile_pbf, y_coord_down=True)

        if LAYER_NAME in decoded:
            for feat in decoded[LAYER_NAME]["features"]:
                merc_geom = project_mvt_geom(feat["geometry"], x_tile, y_xyz, zoom)

                poly_shape = shape(merc_geom)

                features.append({
                    "geometry": poly_shape,
                    **feat["properties"]
                })

    conn.close()

    if not features:
        print("no features found")
        return None


    gdf_3857 = gpd.GeoDataFrame(features, crs="EPSG:3857")
    return gdf_3857.to_crs("EPSG:4326")


land_final_gdf = extract_land_data(MBTILES_PATH, BBOX, ZOOM)

if land_final_gdf is not None:
    land_final_gdf.to_file(OUTPUT_PATH, driver="GPKG", layer="land_polygons")

In [ ]:
land_final_gdf = gdf.to_crs("EPSG:3006")
water_gdf = gdf.to_crs("EPSG:3006")

land_final_gdf.to_file("data/processed/land_data.gpkg", driver="GPKG")
water_gdf.to_file("data/processed/water_data.gpkg", driver="GPKG")